In [10]:
import pandas as pd
from sklearn.decomposition import PCA
import plotly.express as px
from sklearn.preprocessing import StandardScaler

import plotly.io as pio

pio.renderers.default = 'vscode'

In [11]:
data = pd.read_csv("data/Measurements.csv", sep=";")
data.head(5)

,пол,"длина тела, см",длина туловища,длина ноги,"размах рук, см","масса тела, кг",окруж. груд.кл.в покое.,ширина плеч,ширина таза,обхват ягодиц
0,ж,161,48,84,160,"57,3","82,5","30,5",26,"97,5"
1,ж,"151,5",47,NaN,155,"52,4",77,37,28,85
2,м,182,53,NaN,196,71,88,42,33,90
3,м,181,54,NaN,191,84,94,47,32,96
4,м,169,51,NaN,180,"62,8",83,47,47,89


In [12]:
data.describe()

,пол,"длина тела, см",длина туловища,длина ноги,"размах рук, см","масса тела, кг",окруж. груд.кл.в покое.,ширина плеч,ширина таза,обхват ягодиц
count,291,291,289,253,291,291,291,290,291,291
unique,6,99,51,62,85,190,78,42,40,86
top,ж,160,51,79,160,48,83,36,27,92
freq,169,12,31,17,15,8,16,30,33,18


In [13]:
data["пол"].value_counts()

пол
ж      169
м       63
Ж       19
жен     18
М       12
муж     10
Name: count, dtype: int64

In [14]:
alias = {"М": "м", "муж": "м", "м":"м", "Ж": "ж", "жен": "ж", "ж":"ж"}

def gender_format(value: str):
    return alias[value]

data["пол"] = data["пол"].map(gender_format)
data["пол"].value_counts()

пол
ж    206
м     85
Name: count, dtype: int64

In [15]:
required_columns = ["длина тела, см", "масса тела, кг", "окруж. груд.кл.в покое.", "ширина плеч",	"ширина таза", "обхват ягодиц"]

for col in required_columns:
    data[col] = data[col].astype(str).str.replace(",", ".", regex=False)

data = data.astype({col: float for col in required_columns})
data_male, data_female = data[data["пол"] == "м"][required_columns], data[data["пол"] == "ж"][required_columns]
data_male = data_male.dropna()
data_female = data_female.dropna()

# PCA

In [16]:
scaler = StandardScaler()
data_male_scaled = scaler.fit_transform(data_male)

pca_male = PCA(n_components=3)
male_pca_transformed = pca_male.fit_transform(data_male_scaled)
print(f"Объясненная дисперсия по компонентам: {pca_male.explained_variance_ratio_}")
print(f"Объясненная диспресия: {sum(pca_male.explained_variance_ratio_)}")

fig = px.scatter_3d(
    x=male_pca_transformed[:,0],  
    y=male_pca_transformed[:,1],  
    z=male_pca_transformed[:,2],
    height= 500,
    width= 700,
    title="Проекция данных мужчин",
    labels={
        'x': '<b>Главная компонента 1</b>',
        'y': '<b>Главная компонента 2</b>',
        'z': '<b>Главная компонента 3</b>'
    },
    color_continuous_scale='Viridis',  # Красивая цветовая схема
    opacity=0.7,
    size_max=8
)
fig.show()

Объясненная дисперсия по компонентам: [0.70382703 0.11763756 0.08529762]
Объясненная диспресия: 0.9067622041800533


In [17]:
scaler = StandardScaler()
data_female_scaled = scaler.fit_transform(data_female)


pca_female = PCA(n_components=3)
female_pca_transformed = pca_female.fit_transform(data_female_scaled)
print(f"Объясненная дисперсия по компонентам: {pca_female.explained_variance_ratio_}")
print(f"Объясненная диспресия: {sum(pca_female.explained_variance_ratio_)}")

fig = px.scatter_3d(
    x=female_pca_transformed[:,0],  
    y=female_pca_transformed[:,1],  
    z=female_pca_transformed[:,2],
    height= 500,
    width= 700,
    title="Проекция данных женщин",
    labels={
        'x': '<b>Главная компонента 1</b>',
        'y': '<b>Главная компонента 2</b>',
        'z': '<b>Главная компонента 3</b>'
    },
    color_continuous_scale='Viridis',  # Красивая цветовая схема
    opacity=0.7,
    size_max=8
)
fig.show()

Объясненная дисперсия по компонентам: [0.51132391 0.16553583 0.13484227]
Объясненная диспресия: 0.8117020106406295


In [18]:
concat_data = pd.concat([data_male, data_female], axis=0)
gender_labels = ['Мужчины'] * len(data_male) + ['Женщины'] * len(data_female)

scaler = StandardScaler()
concat_data_scaled = scaler.fit_transform(concat_data)


pca_female = PCA(n_components=3)
concat_data_transformed = pca_female.fit_transform(concat_data_scaled)
print(f"Объясненная дисперсия по компонентам: {pca_female.explained_variance_ratio_}")
print(f"Объясненная диспресия: {sum(pca_female.explained_variance_ratio_)}")

df_plot = pd.DataFrame({
    'PC1': concat_data_transformed[:, 0],
    'PC2': concat_data_transformed[:, 1],
    'PC3': concat_data_transformed[:, 2],
    'Пол': gender_labels 
})


fig = px.scatter_3d(
    df_plot,
    x='PC1',
    y='PC2',
    z='PC3',
    color='Пол',  
    height= 500,
    width= 700,
    title="<b>PCA проекция: сравнение мужчин и женщин</b>",
    labels={
        'PC1': '<b>Главная компонента 1</b>',
        'PC2': '<b>Главная компонента 2</b>',
        'PC3': '<b>Главная компонента 3</b>'
    },
    opacity=0.7,
    color_discrete_map={'Мужчины': 'blue', 'Женщины': 'red'} 
)
fig.show()

Объясненная дисперсия по компонентам: [0.59787834 0.14652071 0.13077853]
Объясненная диспресия: 0.8751775837851818


## Выводы

1) Мужчины более однородны, ведь первая компонента объясняет 70,4% дисперсии, а по первым 3 компонентам, объясненная диспресия равна 90.7%. Таким образом телосложение мужчин более предсказуемо.
2) Телосложение женщин более разнообразно, на первую компоненту приходится 51.1% дисперсии, а общая объясненная дисперсия равна 81.2%
3) На графике 3 видивидим большую область наложения облаков точек мужчин и женщин. Таким образом, пол полностью не определяеттелосложение.